# MPR-Agent on `qwen3-4b-thinking` — Kaggle, local weights

The same four-node agent pipeline as `mpr-agent.ipynb` (`DeepSeek-V4-Flash`),
`mpr-agent-gemma-4-31b-it.ipynb`, `mpr-agent-gpt-oss-120b.ipynb` and
`mpr-agent-gemma3-4b-kaggle.ipynb`, driving `Qwen3-4B-Thinking-2507` from
weights this notebook **loads onto the GPU itself**. Implementation of Nguyen et
al., *"A Graph-Based Agent Approach to Numerical Reasoning Question Answering"*
([VLSP 2025](https://aclanthology.org/2025.vlsp-1.29/)).

```
q, C ──▶ [1] SubqueryGenerator   G_sq(q, C)            SQ = {sq_1..sq_k}, k∈[3,5]
                    │ fan-out, one independent call per subquery
         [2] SubqueryAnswerer    A_sq(sq_j, C)         V  = {v_1..v_k}
                    │ fan-in
         [3] Planner             P_n-sample(V,C,q,T)   n = 15 candidate plans
                    │
         [4] EquationExtractor   canonicalise → vote → p* → Execute → a*
```

**All the pipeline logic is reused unchanged** — `agentic/agents.py`,
`program.py`, `prompts.py`, `runner.py`, `scoring.py` are cloned from the repo
and imported. This notebook adds one thing: a **transport** that turns "give me
a completion" into a `model.generate()` call on the loaded Qwen3, and slices the
model's own reasoning trace off the result.

## The one thing that makes this notebook different from its Gemma3 sibling

`Qwen3-4B-Thinking-2507` **always thinks**. Its chat template opens `<think>`
for the model, so every generation is a reasoning trace followed by `</think>`
and only then the answer the pipeline wants. Two consequences run through every
cell below:

1. **Every token budget in the paper has to grow.** `max_tokens_planner=768` is
   the paper's answer budget; on this model those 768 tokens are spent *inside
   the trace* and generation is cut off before `</think>` ever appears. This
   repo's own Qwen3-Thinking eval notebook records exactly that failure — "if
   generation is cut off before the model closes `</think>`, `strip_think` finds
   no closing tag and hands the whole reasoning trace to the parser as if it
   were the program, so a sample that would have been right scores 0 on both
   metrics" — and settles on 2048 new tokens for the *single-prompt* 0-shot
   task. See `THINK_HEADROOM` in section 5.
2. **Wall-clock, not VRAM, is the binding constraint.** The same notebook
   measures ~24 s per generation on a T4. MPR-Agent issues `1 + k + n` ≈ 21 of
   them per sample. Section 6 turns that into an honest projection from your own
   run before you commit a session to it; section 6b is where you spend the
   saving.

Everything else — the adaptive chunked n-sampling, the OOM handling, the
cross-session resume — is carried over from the Gemma3 Kaggle notebook, which
was written against the same 16 GB T4.

## Kaggle setup

* **Accelerator**: GPU (T4 or P100). Only `cuda:0` is used — a second T4 sits idle.
* **Internet**: must be **ON** (Notebook options) for pip, `git clone`, and the
  Hugging Face download.
* No `HF_TOKEN` needed: `unsloth/Qwen3-4B-Thinking-2507` is an ungated mirror of
  `Qwen/Qwen3-4B-Thinking-2507`, and is the exact repo id this project's SFT and
  0-shot notebooks for this model already load.
* **Add Input**: attach your ViNumQA dataset. `/kaggle/input` is read-only, so
  the repo is cloned to `/kaggle/working` instead — see section 2.

## Read this before starting a session: the cost, and the missing baseline

**Cost.** With `use_decomposition=True` and `n_samples=15`, one sample costs
`1 + k + 15` sequential generations (k ≈ 3–5 subqueries) on a single GPU, with
no concurrency available. At the ~24 s/generation this repo measured for this
model on a T4 (`sft-w-reasoning-trace-distill/qwen3-4b-thinking-2507-eval-only.ipynb`,
which notes "~24s each on a T4, so ~3.3h for one pass over the 497 test
questions"), that is:

| configuration | generations/sample | 497 samples @ ~24 s/gen |
|---|---:|---:|
| paper-faithful (decomposition, `n=15`) | ~21 | **~70 h** |
| `n_samples=5` | ~10 | ~33 h |
| `use_decomposition=False`, `n=15` | 15 | ~50 h |
| `use_decomposition=False`, `n=5` | 5 | ~17 h |

A Kaggle session is 12 h. **None of these fit in one.** That is not a reason not
to run it — section 8 resumes across sessions and per-sample checkpointing has
always been on — but it is a reason to decide *now* which row you are running
rather than discovering it at hour 8. The defaults below stay paper-faithful;
section 6 measures your actual s/sample and section 6b is the table to edit.

Note also that ~24 s/generation was measured on the *0-shot* prompt. Two of this
pipeline's four nodes send a longer prompt than that, and the planner sends the
longest one in the pipeline, so treat the table as a floor.

**Baseline.** Unlike the Gemma3 notebook, this one cannot tell you what to
expect. There is **no committed `*_summary.json` for `qwen3-4b-thinking` in
`0-shot/outputs/` or `1-shot/outputs/`** — those two runs exist
(`vsf-0-shot-qwen3-4b-thinking-2507-modal.ipynb`,
`vsf-1-shot-qwen3-4b-thinking-2507-modal.ipynb`) but they run on Modal, write
their outputs to a Modal Volume, and ship with their cell outputs cleared. So
the comparison cell in section 9 will show `NaN` for both ICL rows. That is
correct behaviour, not a broken path, and no number is hardcoded here to paper
over it.

What *is* on record for this model in this repo is a different regime entirely:
the SFT-with-reasoning-trace adapter scores PA 0.6599 / EA 0.6680 under the
repo's older scorer. **That is a fine-tuned checkpoint, not this run** —
MPR-Agent trains nothing and belongs beside the in-context rows. Do not compare
your number to it.

If you want the ICL rows filled in, save either modal notebook's summary to
`0-shot/outputs/0shot_qwen3-4b-thinking_summary.json` in the same shape the
other models use, and re-run section 9.

## Why this notebook defines its own backend instead of using `agentic.backends.LocalBackend`

The package already ships a `LocalBackend`, and `qwen3-4b-thinking` is already
in its `MODEL_REGISTRY` with `thinking=True` — and unlike the Gemma3 case, it
would *load* this checkpoint correctly: `Qwen3ForCausalLM` is exactly what
`AutoModelForCausalLM` maps. Two other things make it the wrong transport on a
Kaggle T4, and both are visible in its source:

```python
# agentic/backends.py, LocalBackend
self._model = AutoModelForCausalLM.from_pretrained(
    self.spec.repo, torch_dtype="auto", device_map="cuda:0",   # (1)
)
...
if n > 1:
    gen_kwargs["num_return_sequences"] = n                     # (2)
```

1. **`torch_dtype="auto"` on a T4.** This checkpoint's config names `bfloat16`,
   and a T4 is sm75 — bf16 has no hardware support there. Unsloth's loader picks
   `float16` on such a card, and offers the 4-bit path this repo's own Kaggle
   notebooks for this exact model already use
   (`FastLanguageModel.from_pretrained(..., load_in_4bit=True)` in both
   `qwen3-4b-thinking-2507-eval-only.ipynb` and
   `qwen3-4b-thinking-2507-stf-w-reasoning-trace.ipynb`).
2. **One `generate()` with `num_return_sequences=15`.** That is the right call
   on a 40 GB+ card and a certain OOM here. Qwen3-4B is 36 layers × 8 KV heads ×
   128 dims, i.e. ~0.14 MB of KV cache per token per sequence. A planner prompt
   of ~4.2k tokens plus a ~2.3k-token thinking generation is ~6.5k tokens, so
   **~0.9 GiB of KV cache per sequence** — times 15 is ~13.7 GiB, on a card that
   has 14.56 GiB total before any weights are loaded.

`LocalBackend`'s docstring is explicit that it is deliberately not Unsloth
because "the 0-shot notebooks this mirrors do not use it either" — true of
`vsf-vinumqa-0-shot-qwen3-4b.ipynb`, and **not** true of any of this repo's
notebooks for the *Thinking* variant, all of which load it with Unsloth and
pinned versions. So rather than patch shared package code that four other
notebooks depend on, this notebook loads the model the way this repo has already
proven works for this exact checkpoint and hands the result to `Runner` through
its existing `client=` parameter. Nothing in `agentic/` changes.

**What it does reuse from the package**, rather than re-implement:
`build_messages("qwen", ...)` for the message shape and `strip_think(tokenizer,
ids)` for the trace slicing — both pure functions, both already covered by
`tests/test_backends.py`. The only genuinely new code here is memory
management.

## 1. Install

Versions pinned to the ones this repo's other `Qwen3-4B-Thinking-2507`
notebooks use — unsloth breaks easily against a mismatched `transformers`.
**Restart the session after this cell** if `transformers` was already imported
in this kernel.

In [ ]:
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Code and data come from two different places

They have to, because **`/kaggle/input` is mounted read-only**. Anything that
needs writing — the clone, the checkpoints, the results — goes to
`/kaggle/working`.

| | where | why |
|---|---|---|
| **repo** (`agentic/`, `scorer.py`, `.git`) | cloned to `/kaggle/working/...` | needs to be written; `.git` is what `find_project_root()` walks up to |
| **dataset** (`test.json`) | your Kaggle Dataset under `/kaggle/input/...` | attached read-only, never written |

Keeping them as one variable is the mistake to avoid: pointing `ROOT` at the
dataset breaks `import agentic`, breaks the scorer lookup, and makes
`find_project_root()` raise — all three, silently, at different moments.

**Attach your dataset** via *Add Input* in the Kaggle sidebar. A dataset
published as `ldhhieu18/vlsp2025` normally mounts at `/kaggle/input/vlsp2025/`,
**not** at `/kaggle/input/datasets/ldhhieu18/vlsp2025/` — so the cell below does
not hardcode either: set `DATA_PATH` explicitly if you know it, otherwise leave
it `None` and the cell finds the file and tells you the real path.

The cell also checks the schema. A ViNumQA split that can be **scored** needs
`qa.program` and `qa.exe_ans`; `private_test.json` carries `qa.question` only,
so if that is what you attached, PA/EA cannot be computed at all and you want to
know now, not after a multi-session run.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# ============================================================== 1. REPO ====
# Cloned into /kaggle/working because it must be WRITABLE: `Runner` writes
# checkpoints under it, and /kaggle/input is read-only (a `git clone` into
# /kaggle/input fails with a permission error).
REPO_URL = "https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText"
REPO_ROOT = Path("/kaggle/working/NumReasoning4VietnameseFinancialText")

if not (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"already cloned: {REPO_ROOT}")

HERE = REPO_ROOT / "notebooks" / "vinumqa" / "graph-agent"
sys.path.insert(0, str(HERE))          # so `import agentic` finds the package

for path in [REPO_ROOT / ".git",
             HERE / "agentic" / "runner.py",
             REPO_ROOT / "notebooks" / "evaluate" / "scorer.py"]:
    assert path.exists(), f"clone incomplete, missing: {path}"
print("repo OK  :", REPO_ROOT)

# ============================================================== 2. DATA ====
# Leave None to auto-discover, or set the path if you already know it, e.g.
#   DATA_PATH = "/kaggle/input/vlsp2025/test.json"
#   DATA_PATH = "/kaggle/input/datasets/ldhhieu18/vlsp2025/test.json"
# Auto-discovery handles both layouts, so None is the safe default: it searches
# /kaggle/input recursively and prints whichever path it actually found.
DATA_PATH = None
DATA_FILENAME = "test.json"

if DATA_PATH is None:
    search_roots = [Path("/kaggle/input")]
    found = [p for root in search_roots if root.exists()
             for p in sorted(root.rglob(DATA_FILENAME))]
    if not found:
        # Fall back to the copy that came with the clone.
        fallback = REPO_ROOT / "datasets" / "ViNumQA" / "origin" / DATA_FILENAME
        assert fallback.exists(), (
            f"no {DATA_FILENAME} under /kaggle/input and none in the clone. "
            f"Attach your dataset via 'Add Input', or set DATA_PATH by hand."
        )
        found = [fallback]
        print(f"no /kaggle/input copy found -- using the repo's own {DATA_FILENAME}")
    if len(found) > 1:
        print(f"several {DATA_FILENAME} found; using the first:")
        for p in found:
            print("   ", p)
    DATA_PATH = found[0]

DATA_PATH = Path(DATA_PATH)
assert DATA_PATH.exists(), f"dataset not found: {DATA_PATH}"
print("dataset  :", DATA_PATH)

# ============================================================ 3. SCHEMA ====
# `load_dataset` takes an absolute path as-is, so DATA_PATH can live anywhere.
with open(DATA_PATH, encoding="utf-8") as handle:
    _samples = json.load(handle)

assert isinstance(_samples, list) and _samples, f"{DATA_PATH} is not a non-empty JSON list"
_first = _samples[0]
_missing_top = {"id", "pre_text", "table", "post_text", "qa"} - set(_first)
assert not _missing_top, f"{DATA_PATH}: samples missing top-level keys {_missing_top}"

_qa = set(_first.get("qa", {}))
SCORABLE = {"program", "exe_ans"} <= _qa
print(f"samples  : {len(_samples)}   qa keys: {sorted(_qa)}")
if SCORABLE:
    print("scorable : yes (qa.program and qa.exe_ans present)")
else:
    print("scorable : NO -- qa carries only", sorted(_qa))
    print("           This looks like a private/unlabelled split. The pipeline")
    print("           will still generate predictions, but PA/EA cannot be")
    print("           computed: every `.score(...)` cell below will be")
    print("           meaningless. Use datasets/ViNumQA/origin/test.json (497")
    print("           labelled samples, shipped with the clone) to measure.")

del _samples

## 3. Load Qwen3-4B-Thinking-2507

`FastLanguageModel.from_pretrained` with 4-bit weights — the same call this
repo's `qwen3-4b-thinking-2507-eval-only.ipynb` and
`qwen3-4b-thinking-2507-stf-w-reasoning-trace.ipynb` already make for this exact
checkpoint on Kaggle: same `model_name`, same `load_in_4bit=True`.

**`LOAD_IN_4BIT = True` is a different call from the Gemma3 notebook's, on
purpose.** There, 16-bit was kept to stay comparable with existing 16-bit rows.
Here every existing row for this model on Kaggle is a 4-bit row, so 4-bit *is*
the comparable setting — and the VRAM it frees goes somewhere the Gemma3 run did
not need it: the KV cache of a long thinking generation.

**The VRAM budget.** A T4 has 14.56 GiB. In 4-bit this model holds ~3.2 GiB,
leaving ~11 GiB. Qwen3-4B is 36 layers × 8 KV heads × 128 head dims, so KV cache
costs `2 (K,V) × 36 × 8 × 128 × 2 bytes = 0.14 MB` per token per sequence. A
planner prompt of ~4.2k tokens plus a ~2.3k-token generation is ~6.5k tokens →
**~0.9 GiB per sequence**. Three at a time is ~2.7 GiB, which fits with room for
the attention workspace; fifteen at a time is ~13.7 GiB, which is the certain
OOM `LocalBackend` would walk into. The backend below starts at 3 and steps
itself down on OOM.

**`MAX_SEQ_LENGTH = 10240`, not the 8192 the sibling notebooks use.** 8192 was
sized for the 0-shot prompt (~4.7k tokens at its longest) plus 2048 new tokens.
This pipeline's planner prompt is that same context **plus every subquery
answer**, and its generation is a thinking trace **plus** the plan. 8192 would
still mostly fit — but the backend clamps `max_new_tokens` to `MAX_SEQ_LENGTH -
prompt_len`, and on a thinking model a clamp that silently cuts the trace short
is exactly what turns a correct sample into a 0 on both metrics. The extra 2k is
a ceiling, not an allocation; it costs nothing at load time.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set before the torch
import, not decoratively: OOM failures on this card are progressive, each one
leaving less free VRAM than the last, and fragmentation from repeated
variable-size allocations is half of that story. The other half is fixed in the
backend cell.

In [ ]:
# CUDA allocator: expandable segments cut the fragmentation that turns one
# planner OOM into the next one. Must be set BEFORE torch initialises CUDA --
# i.e. before the unsloth import below, which is what pulls torch in.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# unsloth must be imported before transformers so its patches take effect.
from unsloth import FastLanguageModel

import gc
import torch

MODEL_REPO = "unsloth/Qwen3-4B-Thinking-2507"   # ungated mirror of Qwen/...
MAX_SEQ_LENGTH = 10240                           # see the markdown above

# 4-bit is the comparable setting for this model in this repo: both Kaggle
# notebooks that run Qwen3-4B-Thinking-2507 load it this way. It also buys the
# ~8 GiB of headroom that a chunked 15-sample thinking generation's KV cache
# needs. Flip to False only on a card with >= 24 GiB, and label the row.
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    load_in_8bit = False,
    full_finetuning = False,
)
FastLanguageModel.for_inference(model)

# ------------------------------------------------------ thinking plumbing --
# Resolved once here rather than per call. `strip_think` in the package does the
# same lookup with the same fallback; this copy exists so the backend can COUNT
# generations that never closed </think> -- the single most important health
# signal on a thinking model, and one the saved traces cannot show you
# afterwards (`Candidate.to_dict` does not keep the raw text).
from agentic.backends import QWEN3_THINK_END_TOKEN_ID

_looked_up = tokenizer.convert_tokens_to_ids("</think>")
THINK_END_ID = (_looked_up if isinstance(_looked_up, int) and _looked_up >= 0
                else QWEN3_THINK_END_TOKEN_ID)

# Qwen3-Thinking-2507's template opens <think> for the model, so the generated
# text starts INSIDE the trace and carries only the closing tag. The slicing
# works either way (strip_think searches from the end and returns everything
# when there is no marker), but the token budgets in section 5 were sized on
# this assumption, so it is worth printing rather than assuming.
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "ping"}],
    add_generation_prompt = True, tokenize = False,
)
PRE_OPENS_THINK = _probe.rstrip().endswith("<think>")

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gib, total_gib = free_bytes / 2**30, total_bytes / 2**30

print("device:", next(model.parameters()).device)
print("dtype :", next(model.parameters()).dtype)
print(f"VRAM  : {total_gib - free_gib:.1f} GiB held, {free_gib:.1f} GiB free "
      f"of {total_gib:.1f} GiB")
print(f"</think> id: {THINK_END_ID}"
      + ("" if THINK_END_ID == QWEN3_THINK_END_TOKEN_ID
         else "  (!! differs from the package constant -- tokenizer changed)"))
print(f"template pre-opens <think>: {PRE_OPENS_THINK}")
if not PRE_OPENS_THINK:
    print("   -> the model must emit BOTH tags itself; budget accordingly.")
print(f"   template tail: {_probe[-60:]!r}")

# The planner is the memory peak: SAMPLE_CHUNK sequences, each holding ~0.9 GiB
# of KV cache for a ~6.5k-token sequence. Below ~2 GiB free it cannot run even
# one at a time, and every sample will fall back to the direct prompt.
if free_gib < 2.0:
    print("\n" + "!" * 78)
    print(f"Only {free_gib:.1f} GiB free -- the planner will OOM even at "
          f"SAMPLE_CHUNK=1.")
    print("Restart the kernel (something else is holding VRAM), or lower")
    print("MAX_SEQ_LENGTH and the token budgets in section 5.")
    print("!" * 78)

## 4. The transport — the only new code in this notebook

`Runner(run_config, client=...)` takes any object with `complete()`,
`sample_n()` and `.usage`; `MultiModelClient` is just the default. Everything in
`agents.py` already calls `self.client.complete(..., model=...)`, so satisfying
that protocol is all it takes to run the unmodified pipeline on local weights.

Two of the seven things below are specific to a *thinking* model; the other five
are carried over from the Gemma3 Kaggle backend, which solved them against the
same card.

* **The reasoning trace is sliced off, and unclosed ones are counted.**
  `strip_think(tokenizer, ids)` is imported from `agentic.backends`, not
  rewritten: it searches from the **end** for the `</think>` token id, so a
  trace that mentions the literal string mid-reasoning is not mistaken for the
  close. When there is no `</think>` at all the whole (truncated) trace comes
  back, which is the honest outcome — it fails to parse and shows up as
  `parse:` in the diagnostics. But *"parse failure"* and *"the budget was too
  small"* are two very different bugs with the same symptom, so this backend
  keeps `unclosed_think` / `total_generations` counters. That ratio is the first
  number to read in section 6, and nothing downstream can recover it: the
  candidate records keep the program and the error, never the raw text.
* **Token budgets must cover the trace, and the clamp is a real risk.**
  `max_new_tokens` is clamped to `MAX_SEQ_LENGTH - prompt_len`. On a non-thinking
  model a too-tight clamp truncates an answer that was mostly written; here it
  truncates a trace whose answer had not started yet, which scores 0. The
  backend prints a one-line warning the first time it clamps below the budget
  asked for, so a silently shortened planner call is visible.
* **One `generate()` at a time.** `agents.py` fans out over subqueries and
  n-samples with a `ThreadPoolExecutor` — correct for HTTP, corrupting for a
  single GPU. `_lock` serialises every forward pass. Note the limit of that
  guarantee: it serialises *compute*, not *allocation*. `max_workers_dataset`
  still opens real threads in `Runner.run`, and each in-flight sample holds its
  own tensors on the same card, so that one is set to 1 in the config cell.
* **Chunked n-sampling, adaptively.** `num_return_sequences=15` over a ~4.2k
  prompt and a ~2.3k thinking generation is ~13.7 GiB of KV cache — more than
  the card. The n samples are generated in chunks of `SAMPLE_CHUNK`, and the
  chunk size *steps down by one on OOM and stays down*. One at a time, not by
  halving: halving takes 3 straight to 1 and never tries 2. Sticky matters too —
  retrying the large size on every planner call would buy a doomed prefill 497
  times over. One prompt's KV cache is still shared within each chunk.
* **Cleanup on the failure path.** `gc.collect()` / `empty_cache()` live in a
  `finally` around the single `generate()` call, not after the sampling loop.
  Cleanup that only runs on success is cleanup that never runs when you need it.
* **The OOM's own traceback is cleared before retrying.** Subtle and
  load-bearing. A live traceback holds one frame per level of `generate()`, and
  each of those frames holds its intermediate tensors, so an `empty_cache()`
  taken while the exception is still bound frees nothing at all. `raise ... from
  None` does *not* fix this — it only hides the context from the printed
  traceback while `__context__` keeps the frames alive. `exc.__traceback__ =
  None` in the handler is what actually releases them, and the reclaim is then
  taken outside the `except` block, where `exc` is unbound.
* **Generated tokens leave the GPU immediately.** `row[prompt_len:]` is a
  *view* — it keeps the whole output tensor alive through decoding. `.tolist()`
  copies to host so the VRAM is actually released, the same thing
  `LocalBackend` does. It matters more here than on a non-thinking model: these
  rows are ~2.3k tokens long, not ~300.

In [ ]:
import threading

from agentic.backends import build_messages, strip_think
from agentic.llm import LLMError, Usage

# `torch.OutOfMemoryError` is the torch 2.x name, `torch.cuda.OutOfMemoryError`
# the older alias. Bind whichever exists, so the except clause below cannot
# itself raise AttributeError on a version bump.
OOM_ERROR = getattr(torch, "OutOfMemoryError", torch.cuda.OutOfMemoryError)


class UnslothQwen3ThinkingBackend:
    """`Backend` protocol over an unsloth-loaded Qwen3-Thinking, for `Runner(client=)`.

    Mirrors `agentic.backends.LocalBackend` in behaviour -- same locking, same
    usage accounting, same budget guard, and the same two pure helpers
    (`build_messages`, `strip_think`) imported rather than copied -- but loads
    nothing itself: it wraps the model/tokenizer already in memory above.

    Two deliberate differences from `LocalBackend`:

    1. n-sampling is chunked and the chunk size is adaptive. One
       `num_return_sequences=15` call is right on a 40GB+ card and fatal on a
       T4 when each sequence carries ~0.9 GiB of KV cache.
    2. It counts generations that never closed `</think>`. Those are budget
       failures, not model failures, and they are indistinguishable from
       ordinary parse failures once the candidate records are written.
    """

    SAMPLE_CHUNK = 3      # starting n-samples per generate(); steps down on OOM

    def __init__(self, model, tokenizer, config, max_seq_length=MAX_SEQ_LENGTH,
                 think_end_id=THINK_END_ID):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.max_seq_length = max_seq_length
        self.think_end_id = think_end_id
        self.usage = Usage()
        self._lock = threading.Lock()
        # Adaptive and STICKY: once a batch size OOMs, that size is never tried
        # again this run. Re-attempting 3 on every planner call would pay for a
        # doomed prefill 497 times over.
        self._chunk = self.SAMPLE_CHUNK
        self.oom_backoffs = 0
        # Thinking-model health counters -- see `unclosed_rate`.
        self.total_generations = 0
        self.unclosed_think = 0
        self.clamped_calls = 0
        self._warned_clamp = False

    @property
    def unclosed_rate(self):
        """Share of generations cut off before the model closed </think>.

        Anything meaningfully above 0 means the token budgets in the config
        cell are too small: those generations hand a truncated reasoning trace
        to the parser and score 0 on both metrics.
        """
        if not self.total_generations:
            return 0.0
        return self.unclosed_think / self.total_generations

    # ---------------------------------------------------------------- memory --
    @staticmethod
    def _free_vram():
        gc.collect()
        torch.cuda.empty_cache()

    def _one_call(self, inputs, prompt_len, gen_kwargs, take):
        """Exactly one generate(). Returns token-id LISTS, already off the GPU.

        The `finally` is the point of this method. When `generate()` raises OOM,
        cleanup placed after the sampling loop never runs, so each failure
        leaves its activations resident and the next sample starts with less
        free VRAM than the last.
        """
        out = None
        try:
            call_kwargs = dict(gen_kwargs)
            if take > 1:
                call_kwargs["num_return_sequences"] = take
            with torch.inference_mode():
                out = self.model.generate(**inputs, **call_kwargs)
            # `.tolist()` copies to host. `row[prompt_len:]` on its own is a
            # view that keeps the whole `out` tensor -- and its VRAM -- alive
            # all the way through decoding, which is the other half of the leak.
            return [row[prompt_len:].tolist() for row in out]
        finally:
            del out
            self._free_vram()

    # ------------------------------------------------------------- generate --
    def _generate(self, system, user, max_tokens, temperature, n):
        """The one place that calls generate(). Caller must hold `_lock`."""
        cfg = self.config
        inputs = self.tokenizer.apply_chat_template(
            build_messages("qwen", system, user),
            add_generation_prompt = True,
            tokenize = True,
            return_tensors = "pt",
            return_dict = True,
        ).to(self.model.device)

        try:
            prompt_len = inputs["input_ids"].shape[-1]
            budget = self.max_seq_length - prompt_len
            if budget <= 0:
                raise LLMError(
                    f"prompt ({prompt_len} tokens) exceeds MAX_SEQ_LENGTH="
                    f"{self.max_seq_length}; nothing left to generate. Reload "
                    f"the model with a larger max_seq_length."
                )
            new_tokens = min(max_tokens, budget)
            if new_tokens < max_tokens:
                # On a thinking model this is not a cosmetic clamp: the answer
                # comes AFTER the trace, so a shortened budget truncates the
                # part that has not been written yet.
                self.clamped_calls += 1
                if not self._warned_clamp:
                    self._warned_clamp = True
                    print(f"  [clamp] prompt={prompt_len} tok leaves only "
                          f"{budget} of the {max_tokens} asked for -- the "
                          f"</think> close may be cut off. Raise "
                          f"MAX_SEQ_LENGTH or lower the token budgets.",
                          flush=True)

            temp = cfg.temperature if temperature is None else temperature
            gen_kwargs = {"max_new_tokens": new_tokens}
            if temp is not None and temp > 0:
                gen_kwargs.update(do_sample=True, temperature=temp, top_p=cfg.top_p)
                if cfg.send_top_k:
                    gen_kwargs["top_k"] = cfg.top_k
            else:
                gen_kwargs["do_sample"] = False

            texts = []
            remaining = max(1, n)
            while remaining > 0:
                take = min(remaining, self._chunk)
                while True:
                    backed_off = False
                    try:
                        gen_only = self._one_call(inputs, prompt_len, gen_kwargs, take)
                    except OOM_ERROR as exc:
                        # Drop the traceback before anything else. It holds one
                        # frame per level of generate(), and every one of those
                        # frames holds its intermediate tensors -- so an
                        # empty_cache() taken while it is alive frees nothing.
                        # `raise ... from None` does NOT do this: it only hides
                        # the context from the printed traceback, leaving
                        # __context__ (and these frames) referenced.
                        exc.__traceback__ = None
                        if take == 1:
                            # Nothing left to step down to. LLMError rather than
                            # a bare OutOfMemoryError is deliberate: the node
                            # records it, this sample degrades to the
                            # direct-prompt fallback, and the run continues.
                            raise LLMError(
                                f"OOM at batch=1 (prompt={prompt_len} tok, "
                                f"max_new_tokens={new_tokens}). Lower the token "
                                f"budgets in the config cell, or set "
                                f"use_decomposition=False to shorten the "
                                f"planner prompt."
                            ) from None
                        backed_off = True
                    if not backed_off:
                        break
                    # Step down by one, NOT by halving. Halving takes 3 straight
                    # to 1 and never tries 2. Each step down is paid for exactly
                    # once per run, because `_chunk` is sticky.
                    take = max(1, take - 1)
                    self._chunk = take
                    self.oom_backoffs += 1
                    # Outside the except block, so `exc` is unbound and the OOM
                    # is collectable: this reclaim actually reclaims.
                    self._free_vram()
                    print(f"  [oom] SAMPLE_CHUNK -> {take} "
                          f"(prompt={prompt_len} tok)", flush=True)

                self.usage.add(
                    prompt_len,
                    sum(len(g) for g in gen_only),
                    prompt_len + sum(len(g) for g in gen_only),
                )
                # A generation with no </think> in it ran out of budget mid
                # trace. Counted, not dropped: dropping it would silently
                # shrink n, and the trace still goes to the parser exactly as
                # the package's own eval notebooks do.
                self.total_generations += len(gen_only)
                self.unclosed_think += sum(
                    1 for g in gen_only if self.think_end_id not in g
                )
                texts += [strip_think(self.tokenizer, g) for g in gen_only]
                remaining -= len(gen_only)
            return texts
        finally:
            del inputs
            self._free_vram()

    # --------------------------------------------------------------- public --
    def complete(self, system, user, model, max_tokens, temperature=None):
        with self._lock:
            outputs = self._generate(system, user, max_tokens, temperature, n=1)
        if not outputs or not outputs[0]:
            raise LLMError("generation returned empty output")
        return outputs[0]

    def sample_n(self, system, user, model, n, max_tokens,
                 temperature=None, max_workers=15):
        if n <= 1:
            return [self.complete(system, user, model, max_tokens, temperature)]
        with self._lock:
            outputs = self._generate(system, user, max_tokens, temperature, n=n)
        usable = [o for o in outputs if o]
        if not usable:
            raise LLMError("n-sampling produced no usable output")
        return usable


print("backend defined:", UnslothQwen3ThinkingBackend.__name__)
print(f"  SAMPLE_CHUNK starts at {UnslothQwen3ThinkingBackend.SAMPLE_CHUNK}, "
      f"steps down one at a time on OOM, floor 1")
print(f"  slicing traces at </think> id {THINK_END_ID}, counting unclosed ones")

## 5. Configuration

**Sampling: the paper and the model card agree, for once.** Paper §5.1 states
`n = 15`, `temperature = 0.6`, `top_p = 0.95`, `top_k = 20` — and those are also
Qwen3-Thinking-2507's own recommended decoding settings. Unlike the Gemma3
notebook, where the paper's values and the model's recommendation pulled in
different directions and one had to be chosen, here there is nothing to trade
off: the row below is simultaneously paper-faithful and model-faithful.

**Token budgets: the one place this run must deviate from the paper.** The
paper's four budgets are answer budgets. This model spends its first ~1–2k
tokens inside `<think>`, so every budget carries a `THINK_HEADROOM` on top.

| node | paper budget | + headroom | why this node |
|---|---:|---:|---|
| subquery gen | 1024 | 2560 | k subqueries; short answer, ordinary reasoning |
| subquery ans | 512 | 2048 | one numeric lookup, but over the full context |
| planner | 768 | 2304 | the longest prompt and the longest reasoning |
| fallback | 512 | 2048 | a plain 0-shot prompt — the setting the repo measured 2048 on |

`THINK_HEADROOM = 1536` is a **starting value, not a measurement**. The right
one is whatever drives `unclosed </think>` in section 6 to near zero, and this
repo's own single-prompt 0-shot run for this model settled on 2048 *total*, so
1536 of headroom on top of the paper's budgets should be comfortable. Raising it
costs nothing on samples that finish early — generation stops at EOS regardless
— so if section 6 reports unclosed generations, raise it and re-run the smoke
cell rather than accepting the loss.

The other reason these matter more here than on an API model: `max_new_tokens`
also sets how much KV cache `generate()` will grow, so every one of them is a
VRAM knob as well as a time one.

In [ ]:
import json
import time

import pandas as pd

from agentic import AgentConfig, RunConfig, Runner
from agentic.runner import candidate_diagnostics, load_dataset, revote, score_frame

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# The exact key this repo uses for this model everywhere else, so the section-9
# lookups and any results table you paste it into line up.
MODEL = "qwen3-4b-thinking"   # label only -- the backend below is what runs

# Room for the reasoning trace, ON TOP of the paper's answer budgets. Raise if
# section 6 reports unclosed </think>; see the table above.
THINK_HEADROOM = 1536

agent_config = AgentConfig(
    model_subquery_gen=MODEL,
    model_subquery_ans=MODEL,
    model_planner=MODEL,
    model_fallback=MODEL,
    # --- paper section 5.1, and also Qwen3-Thinking-2507's own recommended
    #     decoding settings -- the two happen to coincide exactly here ---
    n_samples=15,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    prompt_lang="vi",
    # --- token budgets: paper budget + THINK_HEADROOM ---
    # The paper's numbers are ANSWER budgets. This model answers only after it
    # has closed </think>, so a paper-sized budget is spent entirely inside the
    # trace and the generation is cut off before the program is ever written --
    # which scores 0 on both metrics rather than scoring a worse program.
    max_tokens_subquery_gen=1024 + THINK_HEADROOM,
    max_tokens_subquery_ans=512 + THINK_HEADROOM,
    max_tokens_planner=768 + THINK_HEADROOM,
    max_tokens_fallback=512 + THINK_HEADROOM,
    # --- ours; see README.md ---
    vote_mode="canonical",
    use_prompt_ext=False,
    # False, not the paper-faithful True this notebook started from: measured
    # on DeepSeek-V4-Flash over the full 497, decomposition OFF beat
    # decomposition ON on both PA and EA (see the "Ablation & prompt-fidelity
    # results" table in README.md) and is this repo's actual default for every
    # other model in the comparison table. Flip to True only if you are
    # deliberately reproducing the paper-faithful row instead, and label the
    # result as such -- it is not comparable to the other rows if you do.
    use_decomposition=False,
    # ONE, not the package default of 4. The backend's `_lock` serialises
    # compute, but it does NOT serialise allocation: `Runner.run` really does
    # open a ThreadPoolExecutor, and 4 in-flight samples each hold their own
    # tokenised inputs and their own generate() leftovers on the same card.
    # A thread cannot empty_cache() what another thread is still holding, so
    # 4 workers multiply peak VRAM and fragmentation for no throughput at all.
    max_workers_dataset=1,
    # Irrelevant locally: there is no endpoint to throttle against.
    rpm_limit=None,
    tpm_limit=None,
)

backend = UnslothQwen3ThinkingBackend(model, tokenizer, agent_config)

run_config = RunConfig(
    dataset_path=str(DATA_PATH),        # absolute -- resolved as-is by load_dataset
    output_dir="notebooks/vinumqa/graph-agent/outputs",
    run_name=f"mpr-agent-{MODEL}-kaggle",
    agent=agent_config,
)

runner = Runner(run_config, client=backend)
# k (subqueries per sample) only exists when use_decomposition=True -- with it
# False (this notebook's default now), nodes [1]+[2] never run, so a sample
# costs exactly n_samples generations, not 1 + k + n_samples. Getting this
# wrong doesn't change what gets scored, only what section 6's health-check
# printout claims to expect.
_k = 4 if agent_config.use_decomposition else 0
_gens_per_sample = (1 if agent_config.use_decomposition else 0) + _k + agent_config.n_samples

print(f"MODEL={MODEL!r} -> local weights ({MODEL_REPO}, "
      f"{'4-bit' if LOAD_IN_4BIT else '16-bit'})")
print(f"use_decomposition={agent_config.use_decomposition}  "
      f"n_samples={agent_config.n_samples}")
print(f"generations per sample: {_gens_per_sample}"
      + ("  (1 + k + n_samples, k = subqueries, 3-5)" if agent_config.use_decomposition
         else "  (n_samples only -- decomposition off, nodes [1]+[2] skipped)"))
print(f"planner generate() calls per sample: "
      f"{-(-agent_config.n_samples // backend.SAMPLE_CHUNK)} "
      f"at SAMPLE_CHUNK={backend.SAMPLE_CHUNK}")
print(f"token budgets (think headroom {THINK_HEADROOM}): "
      f"gen={agent_config.max_tokens_subquery_gen} "
      f"ans={agent_config.max_tokens_subquery_ans} "
      f"plan={agent_config.max_tokens_planner} "
      f"fallback={agent_config.max_tokens_fallback}")
print(runner.graph.describe())

## 6. Smoke run — 10 samples

Deliberately 10, not the 30 the API notebooks use: on local weights this is the
cell that tells you whether the full run is feasible **in this session at all**,
and you want that answer in minutes. At ~21 generations/sample it will still
take a while — this is the point at which to go and do something else.

Read these in order, and read the first four *before* looking at PA/EA — a
broken pipeline still prints a complete, plausible-looking summary, because
`equation_extractor` falls back to a plain 0-shot prompt whenever the planner
gives it nothing:

1. `unclosed </think>` — **the thinking-model signal, read it first.** Every
   unclosed generation is a candidate that hands a truncated reasoning trace to
   the parser and is guaranteed to score 0. If this is not near 0, raise
   `THINK_HEADROOM` and re-run this cell; nothing further down is worth reading
   until it is. It is also invisible after the fact: the trace file keeps the
   parsed program and the error, never the raw text.
2. `fallback_rate` — the pipeline health signal. `1.0` means **no sample went
   through MPR-Agent at all**; the PA/EA underneath it are then measuring a
   0-shot prompt, not this method. The cell asserts on this.
3. `mean_usable_candidates` — `0.0` means the planner returned nothing to vote
   over. Same story as above, seen from the other side.
4. `sequences/sample` — the raw count of generations, which should be
   `1 + k + n_samples` ≈ 21. Read this rather than `generate() calls/sample`:
   because the planner's n samples are chunked, one `generate()` call produces
   `SAMPLE_CHUNK` sequences, so the call count in `usage` is only
   `1 + k + ceil(n / SAMPLE_CHUNK)` ≈ 10 even in a perfectly healthy run. The
   cell prints both, with the expected value beside each.
5. `oom backoffs` — 0 is ideal. A few is fine, that is the backend finding its
   batch size. If `SAMPLE_CHUNK` has been driven to 1 and you *still* see OOM
   errors, cut the token budgets or set `use_decomposition=False`.
6. `s/sample` — multiply by 497. Compare against the table in section 0b and
   decide which configuration you are actually running before section 8.
7. `empty_rate` — once the five above are healthy, this one is about the model:
   it means the model emitted something unparseable *after* closing its trace,
   which no runtime fix touches.
8. `mean_consensus` — if it is 1.0, the 15 samples collapsed to one program and
   `n_samples` is pure cost, exactly as measured on `gemma-4-31B-it`. On a
   thinking model at temperature 0.6 this is worth checking before paying for
   n=15 over 497 samples.

In [ ]:
smoke_runner = Runner(
    RunConfig(**{**run_config.__dict__, "run_name": "kaggle-smoke10", "limit": 10}),
    client=backend,
)

started = time.time()
smoke_df = smoke_runner.run(show_progress=True)
elapsed = time.time() - started

smoke_scored, smoke_summary = smoke_runner.score(smoke_df)
usage = smoke_summary["usage"]
n = len(smoke_df)

for key, value in smoke_summary.items():
    print(f"{key:>24}: {value}")
print()
print(f"{'seconds/sample':>24}: {elapsed / n:.1f}")
print(f"{'tokens/sample':>24}: {usage['total_tokens'] / n:.0f}")
# Two different numbers, and confusing them is how a healthy run gets
# diagnosed as broken. `usage.requests` counts generate() CALLS, and the
# planner's 15 samples arrive `SAMPLE_CHUNK` at a time -- so a healthy run
# shows ~10 calls, not ~21. `backend.total_generations` counts sequences,
# which is the ~21 the method actually calls for.
# k (subqueries) only exists when use_decomposition=True -- see the config
# cell. With it False (this notebook's default), a sample is n_samples
# generate() calls/sequences only, not 1 + k + n_samples.
_k = 4 if agent_config.use_decomposition else 0
_base = 1 if agent_config.use_decomposition else 0
expected_calls = _base + _k + -(-agent_config.n_samples // backend._chunk)
expected_seqs = _base + _k + agent_config.n_samples
print(f"{'generate() calls/sample':>24}: {usage['requests'] / n:.1f} "
      f"(expect ~{expected_calls} at SAMPLE_CHUNK={backend._chunk})")
print(f"{'sequences/sample':>24}: {backend.total_generations / n:.1f} "
      f"(expect ~{expected_seqs})")
print(f"{'oom backoffs':>24}: {backend.oom_backoffs} "
      f"(SAMPLE_CHUNK now {backend._chunk})")
print(f"{'unclosed </think>':>24}: {backend.unclosed_think}/"
      f"{backend.total_generations} ({backend.unclosed_rate:.1%})")
print(f"{'budget-clamped calls':>24}: {backend.clamped_calls}")

# ------------------------------------------------- thinking health gate ----
# Read this BEFORE the pipeline gate below, and both before any accuracy
# number. An unclosed </think> is a generation that ran out of tokens while
# still reasoning: strip_think finds no marker, the whole trace goes to the
# parser, and the candidate is dropped at `parse`. It looks exactly like "the
# model cannot write a plan" in the diagnostics, and it is not.
if backend.unclosed_rate > 0.05:
    print("\n" + "!" * 78)
    print(f"{backend.unclosed_rate:.1%} of generations never closed </think>.")
    print("These are BUDGET failures, not model failures, and they score 0.")
    print(f"  -> raise THINK_HEADROOM (now {THINK_HEADROOM}) in the config cell")
    print("     and re-run this cell. Generation stops at EOS anyway, so a")
    print("     larger budget costs nothing on samples that finish early.")
    if backend.clamped_calls:
        print(f"  -> {backend.clamped_calls} call(s) were also clamped by")
        print(f"     MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}: raising THINK_HEADROOM")
        print("     alone will not help those. Raise MAX_SEQ_LENGTH and restart")
        print("     the kernel, or set use_decomposition=False to shorten the")
        print("     planner prompt.")
    print("!" * 78)
else:
    print(f"\nthinking budget OK: {backend.unclosed_rate:.1%} unclosed traces.")

# ---------------------------------------------------- pipeline health gate --
# `fallback_rate` is the honest signal: 1.0 means every sample skipped the
# entire MPR-Agent pipeline and answered from a plain 0-shot prompt, so PA/EA
# above are measuring nothing at all. That is what a planner that always raises
# looks like from out here, and it is indistinguishable from "the model is bad"
# unless you check it.
healthy = (
    smoke_summary["fallback_rate"] <= 0.5
    and smoke_summary["mean_usable_candidates"] > 0
)
if not healthy:
    print("\n" + "!" * 78)
    print("PIPELINE UNHEALTHY -- do NOT start the 497-sample run.")
    print(f"  fallback_rate          = {smoke_summary['fallback_rate']} "
          f"(want <= 0.5)")
    print(f"  mean_usable_candidates = {smoke_summary['mean_usable_candidates']} "
          f"(want > 0)")
    print(f"  sequences/sample       = {backend.total_generations / n:.1f} "
          f"(want ~{expected_seqs})")
    print("Run the diagnostic cell below before changing anything.")
    print("!" * 78)
else:
    print("pipeline healthy: the planner is producing candidates.")

full_n = len(load_dataset(run_config.dataset_path))
projected_h = elapsed / n * full_n / 3600
print(f"\nprojected for {full_n} samples: {projected_h:.1f} h")
if projected_h > 9:
    print(f"  -> will NOT fit one 12h Kaggle session "
          f"(~{projected_h / 8:.1f} sessions at SESSION_BUDGET_H=8).")
    print("     Either pick a cheaper configuration below, or plan on resuming")
    print("     across sessions: checkpointing is per sample, so re-running the")
    print("     full-run cell in a new session skips everything already done.")

### 6b. If the projection is too long — cheaper configurations

Ordered by how much they cost you scientifically, cheapest concession first.
Each is one line; re-run the config cell after editing.

| change | generations/sample | what you lose |
|---|---:|---|
| `use_decomposition=False` | `n` only (drops 1 + k) | nodes [1]+[2]. On `DeepSeek-V4-Flash` this *improved* PA and EA over the full 497 — measured, see `mpr-agent.ipynb`. It also shortens the planner prompt, which is the OOM peak and the slowest prefill. Untested on this model. |
| `n_samples=5` | 1 + k + 5 | vote resolution. Harmless if `mean_consensus` came back 1.0 above; real loss if it did not. |
| both of the above | 5 | the strongest single lever available: ~4× faster than the paper row, and still a clean, labelled cell in the paper's Table 4 ablation grid. |
| `limit=150` in `RunConfig` | unchanged | comparability — a 150-sample number is **not** the same metric as this repo's other 497-sample rows, and must be labelled as such wherever you report it. |

Do **not** reach for `limit` first. A partial run on the full method is a weaker
result than a complete run on a cheaper method, because the latter is still a
clean row in the ablation table the paper's Table 4 defines.

What you should **not** cut on this model is `THINK_HEADROOM`. It looks like a
time knob — fewer tokens, faster generation — but generation stops at EOS
anyway, so a trace that would have finished in 900 tokens costs 900 tokens
whether the cap is 1500 or 2500. All a lower cap does is convert the *long*
traces, which are the hard samples, into guaranteed zeros.

## 7. Read one trace end to end

The point of an agent pipeline over a single prompt is that every intermediate
step is inspectable. Read a few by hand before trusting any aggregate.

One caveat specific to this model: what you see below is **post-`</think>`
text**. The reasoning trace is sliced off in the backend and never written to
the trace file, so a subquery answer that looks abrupt here may have had a page
of reasoning behind it. If you want to see a trace, call
`backend.complete(...)` by hand — the backend returns the sliced text, so to see
the raw form you would have to call `model.generate()` directly.

In [ ]:
with open(smoke_runner.checkpoint_path, encoding="utf-8") as handle:
    traces = json.load(handle)

gold_lookup = {str(s["id"]): s["qa"] for s in load_dataset(run_config.dataset_path)}


def show(record):
    gold = gold_lookup.get(record["id"], {})
    print("=" * 100)
    print(f"id       : {record['id']}")
    print(f"question : {record['question']}")
    print(f"\n[1] subqueries ({len(record['subqueries'])}):")
    for item in record["subqueries"]:
        print(f"      - {item}")
    print("\n[2] answers:")
    for item in record["subquery_answers"]:
        print(f"      * {item['answer'][:150]}")
    candidates = record.get("candidates", [])
    distinct = {c["program"] for c in candidates if c["program"]}
    print(f"\n[3] {len(candidates)} plans sampled -> {len(distinct)} distinct program(s)")
    for program in list(distinct)[:5]:
        print(f"      {program}")
    vote_info = record.get("vote") or {}
    print(
        f"\n[4] clusters={vote_info.get('n_clusters')} "
        f"consensus={vote_info.get('consensus')} fallback={record.get('fallback')}"
    )
    print(f"      p*   : {record['program']}")
    print(f"      gold : {gold.get('program')}")
    print(f"      a*   : {record['answer']}   gold: {gold.get('exe_ans')}")


for record in traces[:3]:
    show(record)

### Where candidates die

Each generated plan either becomes a program or fails at a named stage: `parse`
(not a plan), `transpile` (a plan with no ViNumQA equivalent), `row_lookup`
(`table_*` naming a row the table does not have), or `execute`.

**On a thinking model, `parse` is ambiguous and the backend's counters are what
disambiguate it.** A candidate fails at `parse` either because the model wrote
something that is not a plan, or because it never finished thinking and the
parser was handed a truncated trace. The first is a capability result worth
reporting; the second is a budget bug worth fixing. `backend.unclosed_rate`
tells you which, and the cell below prints it alongside the breakdown.

Compare against the hosted models, where `ok` dominates and the interesting
failures are `transpile` and `row_lookup`.

In [ ]:
from collections import Counter

diagnostics = candidate_diagnostics(smoke_runner.checkpoint_path)

print(f"unclosed </think>: {backend.unclosed_think}/{backend.total_generations} "
      f"({backend.unclosed_rate:.1%}) -- how much of any `parse` share below is "
      f"a budget failure rather than a model failure\n")

if not diagnostics.empty:
    print(diagnostics["stage"].value_counts(normalize=True).round(4).to_string())
    print("\nmost common failures:")
    print(diagnostics[diagnostics["stage"] != "ok"]["error"]
          .value_counts().head(10).to_string())
else:
    # `candidate_diagnostics` builds one row per generated candidate. An empty
    # frame means NO sample produced a single candidate, i.e. the planner never
    # returned a usable plan -- so there is nothing to break down by stage.
    # Indexing the empty frame for "stage" is the `KeyError: 'stage'` you would
    # otherwise get here; the real diagnosis is in the trace file already.
    raw = json.loads(Path(smoke_runner.checkpoint_path).read_text(encoding="utf-8"))
    print(f"NO CANDIDATES across {len(raw)} sample(s).")
    print("The planner returned no usable plan, so there is no stage breakdown.")
    print("This is a pipeline failure, not a model-quality result -- diagnose it")
    print("before reading anything else in this notebook.\n")

    failed = Counter()
    ok_nodes = Counter()
    for record in raw:
        for trace in record.get("traces", []):
            (failed if not trace.get("ok", True) else ok_nodes)[trace["node"]] += 1

    print(f"{'node':<24}{'ok':>6}{'raised':>8}")
    for node in ("subquery_generator", "subquery_answerer", "planner",
                 "equation_extractor"):
        if ok_nodes[node] or failed[node]:
            print(f"{node:<24}{ok_nodes[node]:>6}{failed[node]:>8}")

    messages = Counter(e for record in raw for e in record.get("errors", []))
    if messages:
        print("\nrecorded errors (most common first):")
        for message, count in messages.most_common(8):
            print(f"  [{count}x] {message[:200]}")
    else:
        print("\nNo node raised: the planner returned text, but every plan")
        print("failed to parse into a program. Inspect the raw output directly:")
        print("  raw[0]['traces']  ->  per-node detail")

    blob = " ".join(messages)
    print("\nWhat to check, in order:")

    if backend.unclosed_rate > 0.5:
        # On this model this is the likeliest cause by a wide margin, and it is
        # the one that looks least like itself in the trace file.
        print("  >> MOST GENERATIONS NEVER CLOSED </think>. The token budgets")
        print("     are too small for this model's reasoning traces, so the")
        print("     parser is being handed truncated thinking, not plans.")
        print(f"     Now: THINK_HEADROOM={THINK_HEADROOM}, "
              f"{backend.clamped_calls} call(s) clamped by MAX_SEQ_LENGTH.")
        print("     -> raise THINK_HEADROOM to 2560 in the config cell and")
        print("        re-run the smoke cell. If clamped calls are nonzero,")
        print("        raise MAX_SEQ_LENGTH and restart the kernel too.")
    elif "OutOfMemory" in blob or "OOM" in blob:
        print("  >> CUDA OUT OF MEMORY. This is a VRAM problem, not a prompt or")
        print("     a model problem. The planner is the peak: SAMPLE_CHUNK")
        print("     sequences, each carrying ~0.9 GiB of KV cache for a ~6.5k-")
        print("     token sequence.")
        print(f"     Backend state: SAMPLE_CHUNK={backend._chunk}, "
              f"{backend.oom_backoffs} backoff(s) so far.")
        if backend._chunk > 1:
            print("     -> It has not bottomed out yet. Re-run the smoke cell;")
            print("        the shrunk chunk size is sticky and may already be")
            print("        enough.")
        else:
            print("     -> Already at batch=1 and still OOM. In order:")
            print("        1. use_decomposition=False in the config cell, which")
            print("           drops the subquery answers from the planner")
            print("           prompt and shortens the peak prompt outright.")
            print("        2. THINK_HEADROOM = 1024, which caps how much KV")
            print("           cache generate() will grow (but see the warning")
            print("           in section 6b before doing this).")
            print("        3. MAX_SEQ_LENGTH = 8192 and restart the kernel.")
        print("     Also confirm max_workers_dataset=1: extra Runner threads")
        print("     multiply peak VRAM without buying any throughput here.")
    else:
        print("  1. Prompt length. This backend raises LLMError when the prompt")
        print("     alone exceeds MAX_SEQ_LENGTH. The planner prompt is the")
        print("     longest in the pipeline (context + every subquery answer).")
        print("     If the errors above say 'exceeds MAX_SEQ_LENGTH', reload")
        print("     with a larger max_seq_length, or set")
        print("     use_decomposition=False to drop the subquery answers.")
        print("  2. The system turn. The planner and the fallback are the only")
        print("     nodes that send a system message; the two subquery nodes")
        print("     send system=None. If those two succeeded and only the")
        print("     planner raised, suspect the chat template's system-role")
        print("     handling.")
        print("  3. Empty generations. 'no usable output' means every sample")
        print("     decoded to an empty string AFTER </think> -- the model")
        print("     closed its trace and then wrote nothing. Raise")
        print("     max_tokens_planner.")

    print("\nOne planner prompt, end to end, reproduces it in isolation:")
    print("  from agentic.agents import build_state")
    print("  s = build_state(load_dataset(run_config.dataset_path, limit=1)[0])")
    print("  p = runner.graph.nodes['planner']")
    print("  print(backend.complete(p.prompts.planner_system,")
    print("                         p.build_user_prompt(s), MODEL,")
    print("                         agent_config.max_tokens_planner))")

## 8. Full run — `test.json`, 497 samples, across as many sessions as it takes

**Per-sample checkpointing is already on and always was.**
`RunConfig.checkpoint_every` defaults to `1`, so `Runner` rewrites its traces
file after *every* completed sample, atomically (`.tmp` then `replace`). A CUDA
OOM, a dead kernel, a `KeyboardInterrupt` — none of them can cost you more than
the one sample that was in flight. That part needs no fixing.

What the cell below adds is the thing per-sample checkpointing does *not*
solve — and on this model it is not optional. At the projection section 6 gave
you, the paper-faithful configuration needs **six or more Kaggle sessions**, so
"resume" has to mean *across* sessions:

* **Surviving the end of a session.** The checkpoint lives inside the clone at
  `/kaggle/working/NumReasoning4VietnameseFinancialText/notebooks/vinumqa/graph-agent/outputs/`,
  and the clone is re-created from scratch every session. The cell searches
  `/kaggle/input` for a traces file from a previous run and seeds the checkpoint
  from it before starting.
* **Stopping before the wall, not at it.** Kaggle terminates at 12 h with no
  warning and no opportunity to save. `SESSION_BUDGET_H` stops the run
  deliberately, with time left to save the output. It also refuses to *start* a
  chunk it does not expect to finish inside the budget, using this session's own
  measured s/sample — so it stops early rather than being killed mid-sample.
  `CHUNK_SIZE = 5` bounds the overshoot at about `5 × s/sample`, which on this
  model is ~40 minutes; drop it to 2 if your budget is tight.
* **A stable place to find the file.** Everything is mirrored to
  `/kaggle/working/checkpoints/` after every chunk. Top level, obvious in the
  Output tab, no digging through the clone.
* **Not lying about a partial run.** If it stops early it scores only the
  samples that actually ran — feeding the scorer all 497 with 40 done would mark
  457 empty predictions wrong and report a PA that is about nothing — and it
  writes to a `-partial-NNofMM` filename rather than the canonical
  `{run_name}_summary.json` that the comparison cell further down reads.

### The loop, concretely

1. Run every cell top to bottom. This one stops itself after `SESSION_BUDGET_H`.
2. **Save Version** (or download `/kaggle/working/checkpoints/`) before the
   session ends. Nothing in `/kaggle/working` survives otherwise.
3. New session: **Add Input →** this notebook's saved output, then run every
   cell down to here again. It finds the traces file, prints `restored
   checkpoint from ...`, and picks up at the next unfinished sample.
4. Repeat until it prints `RUN COMPLETE`, at which point it writes the real
   `_results.csv` and `_summary.json`.

`SESSION_BUDGET_H = 8.0` is a starting point, not a measurement — set it to
however long you actually intend to babysit the session, leaving margin to save.

In [ ]:
import io
import re
import shutil

# ============================================================== knobs =======
# Stop cleanly after this long. Kaggle kills the session at 12h with no warning
# and no chance to save, so leave real margin: the cell below also refuses to
# start a chunk it does not expect to finish in time.
SESSION_BUDGET_H = 8.0
# Stop-check granularity. Checkpointing is per-sample regardless
# (`RunConfig.checkpoint_every == 1`); this only bounds how far past the budget
# the run can overshoot -- about `CHUNK_SIZE x s/sample`, which is ~40 min here.
CHUNK_SIZE = 5
# Mirrored here after every chunk. The canonical checkpoint lives inside the
# clone, which is easy to lose track of in the Output tab; this is a stable
# top-level path to download from.
MIRROR_DIR = Path("/kaggle/working/checkpoints")

# =========================================== restore a previous session =====
# If a previous session's traces file was attached as a Kaggle input (Add Input
# -> your earlier notebook version's output), seed the checkpoint from it. The
# clone is fresh every session, so without this the run restarts from zero.
MIRROR_DIR.mkdir(parents=True, exist_ok=True)
if not runner.checkpoint_path.exists():
    kaggle_input = Path("/kaggle/input")
    found = (sorted(kaggle_input.rglob(runner.checkpoint_path.name))
             if kaggle_input.exists() else [])
    if found:
        shutil.copy2(found[0], runner.checkpoint_path)
        print(f"restored checkpoint from {found[0]}")
    else:
        print("no previous checkpoint found -- starting fresh")

# ================================================================ state =====
all_samples = load_dataset(run_config.dataset_path)
runner._load_checkpoint()
done_ids = set(runner._results)
todo = [s for s in all_samples if str(s.get("id", "")) not in done_ids]

print(f"checkpoint : {runner.checkpoint_path}")
print(f"mirror     : {MIRROR_DIR}")
print(f"done       : {len(done_ids)}/{len(all_samples)}")
print(f"remaining  : {len(todo)}")
print(f"budget     : {SESSION_BUDGET_H:.1f} h\n")


# `Runner.run` prints a banner on every call; at one call per chunk that is 100
# lines of noise. Drop those two lines only -- the backend's [oom] and [clamp]
# warnings go to the same stream and must still get through.
_NOISE = re.compile(r"^(?:\d+ sample\(s\) total|Resumed \d+ sample\(s\))")


class _FilteredStdout(io.TextIOBase):
    def __init__(self, target):
        self.target, self._buf = target, ""

    def write(self, text):
        self._buf += text
        while "\n" in self._buf:
            line, self._buf = self._buf.split("\n", 1)
            if not _NOISE.match(line):
                self.target.write(line + "\n")
        return len(text)

    def flush(self):
        self.target.flush()


# ================================================================== run =====
deadline = time.time() + SESSION_BUDGET_H * 3600
session_started = time.time()
done_this_session = 0
stopped_early = False
stop_reason = ""

try:
    from tqdm.auto import tqdm
    bar = tqdm(total=len(todo), desc=run_config.run_name, unit="sample")
except ImportError:
    bar = None

real_stdout = sys.stdout
try:
    for start in range(0, len(todo), CHUNK_SIZE):
        chunk = todo[start:start + CHUNK_SIZE]

        # Do not start a chunk that will not finish in time -- overshooting the
        # budget is how you get killed mid-sample with the session unsaved.
        if done_this_session:
            per_sample = (time.time() - session_started) / done_this_session
            if time.time() + per_sample * len(chunk) > deadline:
                stopped_early = True
                stop_reason = f"budget of {SESSION_BUDGET_H:.1f} h reached"
                break
        elif time.time() > deadline:
            stopped_early = True
            stop_reason = "budget already spent"
            break

        sys.stdout = _FilteredStdout(real_stdout)
        try:
            runner.run(
                samples=chunk,
                show_progress=False,
                on_result=(lambda record: bar.update(1)) if bar else None,
            )
        finally:
            sys.stdout = real_stdout

        done_this_session += len(chunk)
        shutil.copy2(runner.checkpoint_path, MIRROR_DIR / runner.checkpoint_path.name)

        if bar is not None:
            per_sample = (time.time() - session_started) / done_this_session
            left = len(todo) - done_this_session
            bar.set_postfix(
                s_per_sample=f"{per_sample:.0f}",
                eta_h=f"{per_sample * left / 3600:.1f}",
                budget_h_left=f"{max(0.0, deadline - time.time()) / 3600:.1f}",
                unclosed=f"{backend.unclosed_rate:.1%}",
            )
except KeyboardInterrupt:
    stopped_early = True
    stop_reason = "interrupted by hand"
finally:
    sys.stdout = real_stdout
    if bar is not None:
        bar.close()
    if runner.checkpoint_path.exists():
        shutil.copy2(runner.checkpoint_path, MIRROR_DIR / runner.checkpoint_path.name)

# ================================================================ score =====
runner._load_checkpoint()
done_samples = [s for s in all_samples if str(s.get("id", "")) in runner._results]
complete = len(done_samples) == len(all_samples)

# Score only what actually ran. Feeding `to_dataframe` the full 497 while 40 are
# done would score 457 empty predictions as wrong and report a PA that is not
# about this model at all.
df = runner.to_dataframe(done_samples)
scored, summary = runner.score(df)

print(f"\nsession: {(time.time() - session_started) / 3600:.2f} h, "
      f"{done_this_session} sample(s) done here")
print(f"total  : {len(done_samples)}/{len(all_samples)}")
print(f"unclosed </think> this session: {backend.unclosed_think}/"
      f"{backend.total_generations} ({backend.unclosed_rate:.1%})\n")

for key, value in summary.items():
    print(f"{key:>24}: {value}")

if complete:
    results_path, summary_path = runner.save(scored, summary)
    for path in (results_path, summary_path):
        shutil.copy2(path, MIRROR_DIR / path.name)
    print(f"\nRUN COMPLETE -- all {len(all_samples)} samples.")
    print(f"saved: {results_path}\n       {summary_path}")
    print(f"mirrored to {MIRROR_DIR}")
else:
    # Deliberately NOT runner.save(): that writes the canonical
    # `{run_name}_summary.json` which the comparison cell below reads, and a
    # partial number sitting at that path would be indistinguishable from the
    # 497-sample row it is not.
    stem = f"{run_config.run_name}-partial-{len(done_samples)}of{len(all_samples)}"
    partial_csv = runner.output_dir / f"{stem}_results.csv"
    scored.drop(columns=["table_raw"]).to_csv(partial_csv, index=False)
    shutil.copy2(partial_csv, MIRROR_DIR / partial_csv.name)

    print(f"\n{'=' * 78}")
    print(f"STOPPED EARLY: {stop_reason}")
    print(f"The numbers above cover {len(done_samples)} of {len(all_samples)} "
          f"samples. They are NOT the 497-sample row -- do not report them as one.")
    print(f"\nTo continue in a new session:")
    print(f"  1. Save Version (or download) so this session's output is kept.")
    print(f"     The file that matters is:")
    print(f"       {MIRROR_DIR / runner.checkpoint_path.name}")
    print(f"  2. In the new session: Add Input -> this notebook's output.")
    print(f"  3. Run every cell down to and including this one. It finds that")
    print(f"     traces file under /kaggle/input by itself and resumes at "
          f"sample {len(done_samples) + 1}.")
    remaining = len(all_samples) - len(done_samples)
    if done_this_session:
        per_sample = (time.time() - session_started) / done_this_session
        print(f"\n{remaining} left, ~{per_sample:.0f} s/sample "
              f"-> ~{per_sample * remaining / 3600:.1f} h "
              f"(~{per_sample * remaining / 3600 / SESSION_BUDGET_H:.1f} more "
              f"sessions at this budget)")
    print("=" * 78)

### Reading the results frame — which column is the prediction

`scored` and the saved `*_results.csv` follow the column convention
`notebooks/evaluate/scorer.py` expects, and it is **not** the intuitive one:

| column | what it holds |
|---|---|
| `program`, `answer` | **gold**, straight from the dataset |
| `generated_program`, `generated_answer` | **the model's prediction** |
| `pa_score`, `ea_score` | per-row grades |

So `df[["id", "program", "answer"]]` shows you the *dataset*, not your run — it
looks like a plausible list of predictions and will happily convince you the
model nailed every sample. Use `generated_program` / `generated_answer` for
anything you intend to read, quote, or eyeball:

```python
scored[["id", "program", "generated_program", "pa_score", "ea_score"]].head(20)
```

The `show()` trace helper above already reads the checkpoint, where
`record["program"]` **is** the prediction and gold is looked up separately — so
that cell is safe to read as-is. This warning is about the DataFrame and CSV
only.

## 9. Against this repo's existing `qwen3-4b-thinking` rows

Same model, same test set, same scorer — the only thing that changes is the
architecture. This mirrors the paper's Table 2. Rows are read from whatever this
repo has actually saved for this model, so a missing baseline shows as `NaN`
rather than another model's number standing in for it.

**Both ICL rows will be `NaN`.** No `*_summary.json` is committed for
`qwen3-4b-thinking` in `0-shot/outputs/` or `1-shot/outputs/`: both runs exist
as notebooks (`vsf-0-shot-qwen3-4b-thinking-2507-modal.ipynb`,
`vsf-1-shot-qwen3-4b-thinking-2507-modal.ipynb`) but they execute on Modal, write
their outputs to a Modal Volume, and are committed with cleared cell outputs. So
unlike the Gemma3 notebook — which could at least quote its baseline from a
saved cell output and decline to hardcode it — there is no number to quote here
at all. **Do not fill these rows from memory or from the SFT table**; re-run one
of those two notebooks and save its summary to
`0-shot/outputs/0shot_qwen3-4b-thinking_summary.json` in the same shape the
other models use, then re-run this cell.

Note that `qwen3-4b-thinking` is one of the three models this repo *fine-tunes*,
so its SFT and STaNR rows in the root README are a different comparison entirely
— MPR-Agent trains nothing, and belongs beside the in-context rows above.

In [ ]:
ICL_SETTINGS = {
    "0-shot": REPO_ROOT / f"notebooks/vinumqa/0-shot/outputs/0shot_{MODEL}_summary.json",
    "1-shot": REPO_ROOT / f"notebooks/vinumqa/1-shot/outputs/1shot_{MODEL}_summary.json",
}

rows = []
for label, path in ICL_SETTINGS.items():
    entry = {"method": f"{MODEL}, {label}", "PA": None, "EA": None}
    if path.exists():
        with open(path, encoding="utf-8") as handle:
            blob = json.load(handle)
        entry["PA"] = round(blob["program_accuracy"], 4)
        entry["EA"] = round(blob["execution_accuracy"], 4)
    else:
        print(f"no saved summary for {label}: {path.relative_to(REPO_ROOT)}")
    rows.append(entry)

# A resumed run may still be partway through. The ICL rows above are full
# 497-sample numbers, so putting an unlabelled 40-sample number next to them in
# the same table is the one way this notebook could still mislead you after the
# run cell warned. Tag it in the row name itself, where it travels with the
# number if the table gets copied out.
suffix = "" if complete else f"  [PARTIAL {summary['n']}/{len(all_samples)}]"

rows += [
    {
        "method": f"{MODEL}, MPR-Agent{suffix}",
        "PA": round(summary["program_accuracy"], 4),
        "EA": round(summary["execution_accuracy"], 4),
    },
    {
        "method": f"{MODEL}, MPR-Agent (oracle@{agent_config.n_samples}){suffix}",
        "PA": round(summary["oracle_pa"], 4),
        "EA": round(summary["oracle_ea"], 4),
    },
]

if not complete:
    print(f"WARNING: the two MPR-Agent rows cover {summary['n']} of "
          f"{len(all_samples)} samples and are NOT comparable with the "
          f"0-/1-shot rows above.")

pd.DataFrame(rows)

## 10. Re-vote offline — free, no GPU time

Every candidate is kept on disk with its program and executed value
(`keep_all_candidates`), so changing *how the winner is chosen* costs nothing.
On this run that matters more than anywhere else in the repo: re-running the
pipeline is days of your own GPU, while re-voting is seconds of CPU.

In [ ]:
samples = load_dataset(run_config.dataset_path)
rows = []
for mode in ("canonical", "symbolic"):
    _, mode_summary = score_frame(revote(runner.checkpoint_path, samples, mode=mode))
    rows.append(
        {
            "vote_mode": mode,
            "PA": round(mode_summary["program_accuracy"], 4),
            "EA": round(mode_summary["execution_accuracy"], 4),
        }
    )
pd.DataFrame(rows)

## What was and was not verified before you ran this

Stated plainly so you know where to look first if something breaks.

**Verified without a GPU** (read from this repo, before this notebook was
handed over):

* `unsloth/Qwen3-4B-Thinking-2507`, `FastLanguageModel.from_pretrained`,
  `load_in_4bit=True` and `transformers==4.56.2` / `trl==0.22.2` are exactly
  what `sft-w-reasoning-trace-distill/qwen3-4b-thinking-2507-eval-only.ipynb`
  and `...-stf-w-reasoning-trace.ipynb` already use for this checkpoint on
  Kaggle. Nothing here is a new loading recipe.
* `strip_think` and `build_messages` are imported from `agentic/backends.py`,
  not re-implemented, and both are already covered by `tests/test_backends.py`.
  The `</think>` id is cross-checked against the loaded tokenizer at load time
  rather than trusted.
* `Runner(run_config, client=...)` accepts an arbitrary client and passes it
  straight to `build_default_graph`, and `agents.py` only ever calls
  `client.complete(...)` and `client.sample_n(...)` — the two methods this
  backend implements.
* `Candidate.to_dict` does not keep the raw generated text, which is why the
  unclosed-`</think>` count has to be taken in the backend as it generates:
  after the fact, a truncated trace and an unparseable plan are the same
  `parse:` row.

**Not verified** — no GPU was available when this was written:

* The actual `from_pretrained` call, the pinned install set against Kaggle's
  current image, and anything about VRAM. `SAMPLE_CHUNK = 3` is arithmetic from
  the KV-cache size, not a measurement; it steps itself down on OOM.
* `THINK_HEADROOM = 1536`. This is the one number most likely to need changing,
  and section 6 measures it for you. The evidence behind the starting value is
  that this repo's single-prompt 0-shot run for this model used 2048 tokens
  total and reported its unclosed count as near zero.
* Every runtime number. The ~24 s/generation and the hours in the table at the
  top are carried over from this repo's own 0-shot measurement on a T4 for this
  model, on a *shorter* prompt than the planner sends — treat them as a floor,
  and use your own smoke-run projection instead.

**If the load cell fails**, the pinned versions are the first suspect — Kaggle
updates its base image and `transformers==4.56.2` may need moving in step with
whatever unsloth currently requires. Check Unsloth's own current Qwen3 notebook
(<https://unsloth.ai/docs/get-started/unsloth-notebooks>) and match it, then
update the pins here and in the two `qwen3-4b-thinking-2507-*` notebooks
together.